[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 Hard: INT8 Quantized Linear

Implement a **post-training quantized linear layer** using INT8 weights.

### Signature
```python
class Int8Linear(nn.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Quantization (per-channel)
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. Store as `register_buffer` (not trainable)
4. Forward: dequantize (`int8.float() * scale`) then matmul

In [2]:
import torch
import torch.nn as nn

In [24]:
# ✏️ YOUR IMPLEMENTATION HERE

class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        scale = weight.abs().max(dim=1,keepdim=True).values/127
        self.register_buffer('weight_int8',torch.round(weight/scale).clamp(min=-128,max=127).to(torch.int8))
        self.register_buffer('scale',scale)
        self.bias = nn.Parameter(bias) if bias is not None else None

    def forward(self, x):
        weights = self.weight_int8 * self.scale
        out = x @ weights.T
        if self.bias is not None:
            out += self.bias
        return out

In [25]:
# 🧪 Debug
w = torch.randn(8, 4)
q = Int8Linear(w)
x = torch.randn(2, 4)
print('Output:', q(x).shape)
print('dtype:', q.weight_int8.dtype)
print('Max quant error:', (w - q.weight_int8.float() * q.scale).abs().max().item())

Output: torch.Size([2, 8])
dtype: torch.int8
Max quant error: 0.006760120391845703


In [26]:
# ✅ SUBMIT
from torch_judge import check
check('int8_quantization')


🧪 Testing: INT8 Quantized Linear (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Weight is int8 (2.5ms)
  ✅ [2/5] Values in [-128, 127] (2.4ms)
  ✅ [3/5] Dequantized close to original (5.0ms)
  ✅ [4/5] Forward output shape (2.4ms)
  ✅ [5/5] Weight is buffer not parameter (1.0ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (13.3ms total)
  Progress saved. Run status() to see your dashboard.

